##**Time-Series Feature Engineering for Energy Forecasting**

### **📌 Notebook Objective**

This notebook focuses on feature engineering to transform raw smart-meter and weather data into machine-learning-ready datasets for hourly and daily energy forecasting.

The goal is to encode temporal dependencies, behavioral patterns, and physical drivers of energy consumption into structured features.

### **📌 Feature Engineering Strategy**

The feature engineering approach follows industry-standard time-series forecasting practices, combining:
* Temporal features
* Lagged demand features
* Rolling statistics
* Weather and calendar variables

This mirrors how forecasting pipelines are deployed in real-world energy management systems.

### **📌 Time-Based Features**

The following time-derived features are created:
* Hour of day
* Day of week
* Weekend indicator
* Month of year

These features capture daily, weekly, and seasonal consumption patterns inherent in building energy usage.

### **📌 Lag Features**

Lagged demand variables are introduced, including:
* Previous hour demand (t-1)
* Same hour previous day (t-24)
* Previous day and week demand (for daily models)

Lag features are critical for modeling temporal dependence, which is a defining characteristic of short-term load forecasting.

### **📌 Rolling Statistics**

Rolling window statistics are computed to capture recent demand trends:
* 24-hour rolling mean
* 24-hour rolling standard deviation

These features help smooth noise and represent recent operational context.

### **📌 Hourly and Daily Dataset Preparation**

Two separate datasets are prepared:
* Hourly dataset for short-term (next-hour to next-day) forecasting
* Daily aggregated dataset for day-ahead energy demand forecasting

This separation reflects how utilities operate multiple forecasting horizons in practice.

### **📌 Key Takeaways**
* Raw data is transformed into structured, ML-ready features
* Feature design captures temporal, behavioral, and physical effects
* Separate hourly and daily datasets improve modeling clarity and performance

This notebook bridges the gap between data understanding and predictive modeling.

**Import Necessary Libraries**

In [ ]:
import pandas as pd
import numpy as np

**Load Cleaned Raw Data**

In [ ]:
DATA_PATH = "/content/db_building_A.csv"

df = pd.read_csv(DATA_PATH)
df["DATE"] = pd.to_datetime(df["DATE"], format='mixed', dayfirst=False)
df = df.sort_values("DATE")
df = df.set_index("DATE")

df.head()

,ENERGY,HDD18_3,CDD0,CDD10,PRECTOT,RH2M,T2M,T2M_MIN,T2M_MAX,ALLSKY,HOLIDAY
DATE,,,,,,,,,,,
2016-01-01 00:00:00,NaN,11.32,6.98,0.0,10.7,90.97,6.02,1.74,12.23,1.7,1
2016-01-01 01:00:00,113.415,11.32,6.98,0.0,10.7,90.97,6.02,1.74,12.23,1.7,1
2016-01-01 02:00:00,112.431,11.32,6.98,0.0,10.7,90.97,6.02,1.74,12.23,1.7,1
2016-01-01 03:00:00,111.517,11.32,6.98,0.0,10.7,90.97,6.02,1.74,12.23,1.7,1
2016-01-01 04:00:00,117.240,11.32,6.98,0.0,10.7,90.97,6.02,1.74,12.23,1.7,1


**Handle Missing Values**

In [ ]:
df = df.interpolate(method="time")
df = df.dropna()

df.isna().sum()

,0
ENERGY,0
HDD18_3,0
CDD0,0
CDD10,0
PRECTOT,0
RH2M,0
T2M,0
T2M_MIN,0
T2M_MAX,0
ALLSKY,0


**Time-Based Features**

In [ ]:
df["hour"] = df.index.hour
df["dayofweek"] = df.index.dayofweek
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
df["month"] = df.index.month

**Lag Features**

In [ ]:
df["lag_1"] = df["ENERGY"].shift(1)
df["lag_24"] = df["ENERGY"].shift(24)

**Rolling Statistics**

In [ ]:
df["rolling_mean_24"] = df["ENERGY"].rolling(window=24).mean()
df["rolling_std_24"] = df["ENERGY"].rolling(window=24).std()

**Drop Initial NaNs from Feature Creation**

In [ ]:
df_hourly = df.dropna()

df_hourly.head()

,ENERGY,HDD18_3,CDD0,CDD10,PRECTOT,RH2M,T2M,T2M_MIN,T2M_MAX,ALLSKY,HOLIDAY,hour,dayofweek,is_weekend,month,lag_1,lag_24,rolling_mean_24,rolling_std_24
DATE,,,,,,,,,,,,,,,,,,,
2016-01-02 01:00:00,115.614,12.83,5.47,0.0,0.32,84.78,4.78,2.52,8.42,1.99,1,1,5,1,1,114.112,113.415,114.840375,2.015072
2016-01-02 02:00:00,113.107,12.83,5.47,0.0,0.32,84.78,4.78,2.52,8.42,1.99,1,2,5,1,1,115.614,112.431,114.868542,1.984421
2016-01-02 03:00:00,116.171,12.83,5.47,0.0,0.32,84.78,4.78,2.52,8.42,1.99,1,3,5,1,1,113.107,111.517,115.062458,1.866564
2016-01-02 04:00:00,115.038,12.83,5.47,0.0,0.32,84.78,4.78,2.52,8.42,1.99,1,4,5,1,1,116.171,117.240,114.970708,1.808077
2016-01-02 05:00:00,114.848,12.83,5.47,0.0,0.32,84.78,4.78,2.52,8.42,1.99,1,5,5,1,1,115.038,114.948,114.966542,1.808247


**Save Hourly Feature Dataset**

In [ ]:
import os

HOURLY_OUT = "../data/processed/hourly_features.csv"

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(HOURLY_OUT), exist_ok=True)

df_hourly.to_csv(HOURLY_OUT)

print("Saved:", HOURLY_OUT)

Saved: ../data/processed/hourly_features.csv


**Daily Aggregation (For Daily Forecasting)**

In [ ]:
df_daily = df.resample("D").agg({
    "ENERGY": "sum",
    "T2M": "mean",
    "RH2M": "mean",
    "CDD10": "sum",
    "HDD18_3": "sum",
    "ALLSKY": "mean",
    "HOLIDAY": "max"
})

**Daily Time Features**

In [ ]:
df_daily["dayofweek"] = df_daily.index.dayofweek
df_daily["is_weekend"] = (df_daily["dayofweek"] >= 5).astype(int)
df_daily["month"] = df_daily.index.month

**Daily Lag Features**

In [ ]:
df_daily["lag_1"] = df_daily["ENERGY"].shift(1)
df_daily["lag_7"] = df_daily["ENERGY"].shift(7)

**Drop NaNs & Save Daily Dataset**

In [ ]:
df_daily = df_daily.dropna()

DAILY_OUT = "../data/processed/daily_features.csv"
df_daily.to_csv(DAILY_OUT)

print("Saved:", DAILY_OUT)


Saved: ../data/processed/daily_features.csv
